# AAPL Stock Price Prediction — Part 1: Data

Download AAPL OHLCV data from Yahoo Finance and compute technical indicators using TA-Lib. The final CSV (`AAPL.csv`) is the input for all subsequent notebooks.

**Data source:** Yahoo Finance via `yfinance` (`period='max'`, unadjusted prices)  
**Date range:** 1980-12-12 to present  
**Missing data:** TA-Lib indicators produce NaN during the warm-up period (longest: 100 bars for SMA_100)  
**Features saved:** Close + 8 technical indicators (RSI, Stochastic %K/%D, EMA_20, MACD line/signal/diff, Williams %R)

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib

# ---------------------------------------------------------------
# 1. Download data
# ---------------------------------------------------------------
TICKER = "AAPL"

data = yf.download(TICKER, period="max", group_by="column", auto_adjust=False)

# Flatten MultiIndex if present (happens with some yfinance versions)
if isinstance(data.columns, pd.MultiIndex):
    if TICKER in data.columns.get_level_values(-1):
        data = data.xs(TICKER, axis=1, level=-1)
    elif TICKER in data.columns.get_level_values(0):
        data = data.xs(TICKER, axis=1, level=0)
    else:
        data.columns = data.columns.get_level_values(0)

print(f"Downloaded {len(data)} rows  |  {data.index[0].date()} – {data.index[-1].date()}")
data.head()

[*********************100%***********************]  1 of 1 completed

Downloaded 11399 rows  |  1980-12-12 – 2026-03-06


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
1980-12-12,0.098297,0.128348,0.128906,0.128348,0.128348,469033600
1980-12-15,0.093169,0.121652,0.122210,0.121652,0.122210,175884800
1980-12-16,0.086331,0.112723,0.113281,0.112723,0.113281,105728000
1980-12-17,0.088468,0.115513,0.116071,0.115513,0.115513,86441600
1980-12-18,0.091033,0.118862,0.119420,0.118862,0.118862,73449600


In [2]:
# ---------------------------------------------------------------
# 2. Compute technical indicators with TA-Lib
# ---------------------------------------------------------------

def arr1d(series) -> np.ndarray:
    """Convert a pandas Series (or single-column DataFrame) to a 1-D float array."""
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)

idx   = data.index
close = arr1d(data["Close"])
high  = arr1d(data["High"])
low   = arr1d(data["Low"])

# RSI (14)
data["RSI_14"] = pd.Series(talib.RSI(close, timeperiod=14), index=idx)

# Stochastic %K / %D  (14, 3, 3)
k, d = talib.STOCH(high, low, close, fastk_period=14, slowk_period=3, slowd_period=3)
data["Stoch_%K"] = pd.Series(k, index=idx)
data["Stoch_%D"] = pd.Series(d, index=idx)

# EMA (20)
data["EMA_20"] = pd.Series(talib.EMA(close, timeperiod=20), index=idx)

# MACD (12, 26, 9)
macd, macdsig, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
data["MACD_Line"]   = pd.Series(macd,     index=idx)
data["MACD_Signal"] = pd.Series(macdsig,  index=idx)
data["MACD_Diff"]   = pd.Series(macdhist, index=idx)

# Williams %R (14)
data["Williams_%R"] = pd.Series(talib.WILLR(high, low, close, timeperiod=14), index=idx)

print("Indicators computed. NaN counts:")
indicator_cols = ["RSI_14", "Stoch_%K", "Stoch_%D", "EMA_20",
                  "MACD_Line", "MACD_Signal", "MACD_Diff", "Williams_%R"]
print(data[indicator_cols].isna().sum())

Indicators computed. NaN counts:
Price
RSI_14         14
Stoch_%K       17
Stoch_%D       17
EMA_20         19
MACD_Line      33
MACD_Signal    33
MACD_Diff      33
Williams_%R    13
dtype: int64


In [3]:
# ---------------------------------------------------------------
# 3. Save to CSV
# ---------------------------------------------------------------
FEATURES = ["Close"] + indicator_cols

model_df = data[FEATURES].copy()
model_df.to_csv(f"{TICKER}.csv")

print(f"Saved {TICKER}.csv  ({len(model_df)} rows, {len(FEATURES)} columns)")
model_df.tail()

Saved AAPL.csv  (11399 rows, 9 columns)


Price,Close,RSI_14,Stoch_%K,Stoch_%D,EMA_20,MACD_Line,MACD_Signal,MACD_Diff,Williams_%R
Date,,,,,,,,,
2026-03-02,264.720001,48.217792,47.173805,58.596972,266.689190,0.840808,0.936171,-0.095363,-62.515140
2026-03-03,263.750000,47.231167,35.449532,47.296596,266.409267,0.499486,0.848834,-0.349349,-66.437507
2026-03-04,262.519989,45.947283,35.089350,39.237562,266.038860,0.128255,0.704718,-0.576463,-65.779303
2026-03-05,260.290009,43.631621,30.403390,33.647424,265.491350,-0.341947,0.495385,-0.837332,-76.573019
2026-03-06,257.459991,40.819974,23.953700,29.815480,264.726459,-0.932198,0.209869,-1.142067,-85.786579
